In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Imports
import pandas as pd
import numpy as np
import joblib

In [4]:
# Load Trained Fertilizer Model
model = joblib.load("/content/drive/MyDrive/Coffee/Dev/C3/fertilizer model/coffee_fertilizer_model_per_ha.pkl")
print("Fertilizer model loaded")

Fertilizer model loaded


In [5]:
# Define Scheduling Rules (AGRONOMIC LOGIC)
# Annual fertilizer split by growth stage
STAGE_SPLIT = {
    "vegetative": 0.30,
    "flowering": 0.25,
    "fruiting": 0.30,
    "post-harvest": 0.15
}

In [6]:
# Growth Stage -> Preferred Months Mapping
STAGE_MONTHS = {
    "vegetative": [2, 3],
    "flowering": [4, 5],
    "fruiting": [7, 8],
    "post-harvest": [10, 11]
}

In [7]:
# Rainfall Adjustment Rules
def rainfall_adjustment(monthly_rainfall):
    """
    High rainfall → split dose
    """
    if monthly_rainfall > 200:
        return 0.9  # reduce per-application dose
    return 1.0

In [8]:
# Labor Estimation Function
# Assumption (explicit):
# 1 laborer can apply fertilizer to 0.5 ha/day

def estimate_labor(area_ha):
    laborers = int(np.ceil(area_ha / 0.5))
    return laborers


In [9]:
# Core Scheduling Function
def generate_fertilizer_schedule(input_row, area_ha=1.0):
    """
    input_row: one-row DataFrame with features required by the model
    """

    # Predict total annual NPK (per ha)
    N_total, P_total, K_total = model.predict(input_row)[0]

    growth_stage = input_row["growth_stage"].values[0]
    rainfall = input_row["rainfall_mm"].values[0]

    stage_fraction = STAGE_SPLIT[growth_stage]
    rain_factor = rainfall_adjustment(rainfall)

    N_apply = N_total * stage_fraction * rain_factor
    P_apply = P_total * stage_fraction * rain_factor
    K_apply = K_total * stage_fraction * rain_factor

    months = STAGE_MONTHS[growth_stage]
    per_month = len(months)

    schedule = []

    for m in months:
        schedule.append({
            "month": m,
            "N_kg_per_ha": round(N_apply / per_month, 1),
            "P_kg_per_ha": round(P_apply / per_month, 1),
            "K_kg_per_ha": round(K_apply / per_month, 1),
            "laborers_needed": estimate_labor(area_ha)
        })

    return pd.DataFrame(schedule)


In [10]:
# Example Input
# This simulates a real call from:
# -Yield model
# -Soil data
# -Weather data

example_input = pd.DataFrame([{
    "predicted_yield_kg_per_ha": 1800,
    "soil_n": 45,
    "soil_p": 25,
    "soil_k": 110,
    "rainfall_mm": 220,
    "avg_temperature_c": 24,
    "month": 4,
    "growth_stage": "flowering",
    "coffee_variety": "Arabica"
}])

In [11]:
# Generate Fertilizer Schedule
schedule_df = generate_fertilizer_schedule(
    example_input,
    area_ha=1.0
)

schedule_df

,month,N_kg_per_ha,P_kg_per_ha,K_kg_per_ha,laborers_needed
0,4,18.7,5.9,18.3,2
1,5,18.7,5.9,18.3,2


In [12]:
# Export Schedule
schedule_df.to_csv(
    "/content/drive/MyDrive/Coffee/Dev/C3/fertilizor shedule and labor/fertilizer_application_schedule.csv",
    index=False
)

print("Fertilizer schedule saved")

Fertilizer schedule saved
